# Chapter 18: Particle Filters

<a href="../lite/lab/index.html?path=ch18_particle_filters.ipynb" target="_blank" style="display:inline-block;padding:8px 18px;background:#1976d2;color:white;border-radius:5px;text-decoration:none;font-weight:bold;font-size:0.95em;">&#9654; Open in JupyterLite: run and edit this notebook</a>

*Runs entirely in your browser, no installation required.*

**How to use:** Edit the parameter values in each cell and re-run it to explore.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Imagine releasing 1000 tiny robot clones into the world. Each one moves slightly differently (random noise). After each sensor reading, the clones that match well survive; the others die and are replaced by copies of survivors. After a few steps, all surviving clones cluster around the true position. Congratulations, you just invented the particle filter.

The **particle filter** (also called **Sequential Monte Carlo**) is the most flexible state estimation algorithm. It makes no assumptions about the shape of the distribution: it can handle multimodal beliefs, nonlinear dynamics, and non-Gaussian noise. The tradeoff is computational cost, since you need many particles for accuracy, but for problems where Kalman filters break down, particles are often the only practical option.

```{admonition} What you will build
:class: tip

- Implement a particle filter with importance sampling and low variance resampling
- Watch 500 particles converge from a uniform scatter to cluster around the true robot position
- Handle the kidnapped robot problem by injecting random particles
- Understand particle degeneracy and effective sample size

**Real world application:** Particle filters power Monte Carlo Localization (MCL), the algorithm used by most mobile robots for global localization. After this chapter, you can localize a robot that has no idea where it started.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **nav2_amcl (ROS 2)** | The standard particle filter for robot localization in ROS 2 |
| **particles (Python)** | Lightweight sequential Monte Carlo library |
| **mrpt::slam::CMonteCarloLocalization2D** | C++ particle filter for 2D localization |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## Opening Demo: 500 Particles Converge

A robot is somewhere in a 2D room with 4 landmarks. We scatter 500 particles uniformly. After a few steps of moving and sensing, watch the cloud collapse from a uniform scatter to a tight cluster around the true position.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_demo = 500       # number of particles          (try 100, 500, 2000)
room_size        = 20.0      # room is [0, room_size] x [0, room_size]
landmarks_demo   = np.array([[2, 2], [2, 18], [18, 2], [18, 18]])  # 4 corners
range_noise_demo = 1.0       # sensor noise std (m)         (try 0.3, 1.0, 3.0)
motion_noise_demo = 0.5      # motion noise std (m)         (try 0.1, 0.5, 1.5)
n_steps_demo     = 12        # steps to simulate            (try 5, 12, 25)
true_start_demo  = np.array([7.0, 14.0, np.pi/6])  # [x, y, theta]
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

def range_to_landmarks(state, landmarks):
    """Compute range from state [x, y, ...] to each landmark."""
    dx = landmarks[:, 0] - state[0]
    dy = landmarks[:, 1] - state[1]
    return np.sqrt(dx**2 + dy**2)

def move_particle(p, v, dt, noise_std):
    """Move a particle [x, y, theta] forward."""
    theta = p[2] + np.random.normal(0, 0.1)
    x = p[0] + v * np.cos(theta) * dt + np.random.normal(0, noise_std)
    y = p[1] + v * np.sin(theta) * dt + np.random.normal(0, noise_std)
    return np.array([x, y, theta])

def weight_particle(p, z_measured, landmarks, noise_std):
    """Compute weight of a particle given measured ranges."""
    z_predicted = range_to_landmarks(p, landmarks)
    diff = z_measured - z_predicted
    log_w = -0.5 * np.sum((diff / noise_std)**2)
    return np.exp(log_w)

def low_variance_resample(particles, weights):
    """Low-variance resampling (systematic resampling)."""
    N = len(weights)
    positions = (np.arange(N) + np.random.uniform()) / N
    cumsum = np.cumsum(weights)
    indices = np.searchsorted(cumsum, positions)
    return particles[indices].copy()

# Initialize particles uniformly
particles = np.zeros((n_particles_demo, 3))
particles[:, 0] = np.random.uniform(0, room_size, n_particles_demo)
particles[:, 1] = np.random.uniform(0, room_size, n_particles_demo)
particles[:, 2] = np.random.uniform(-np.pi, np.pi, n_particles_demo)

true_state = true_start_demo.copy()
v_demo = 1.5
dt_demo_pf = 1.0

# Collect snapshots
snap_steps = [0, 1, 3, 6, n_steps_demo - 1]
snap_steps = sorted(set(s for s in snap_steps if s < n_steps_demo))
snapshots_pf = [('Step 0 (initial)', particles.copy(), true_state.copy())]

for step in range(n_steps_demo):
    # Move true robot
    true_state[0] += v_demo * np.cos(true_state[2]) * dt_demo_pf
    true_state[1] += v_demo * np.sin(true_state[2]) * dt_demo_pf
    true_state[2] += 0.15  # slight turn

    # Move particles
    for i in range(n_particles_demo):
        particles[i] = move_particle(particles[i], v_demo, dt_demo_pf, motion_noise_demo)

    # Sense: range to landmarks
    z_true = range_to_landmarks(true_state, landmarks_demo)
    z_meas = z_true + np.random.normal(0, range_noise_demo, len(landmarks_demo))

    # Weight particles
    weights = np.array([weight_particle(p, z_meas, landmarks_demo, range_noise_demo)
                        for p in particles])
    weights += 1e-300  # avoid all-zero
    weights /= weights.sum()

    # Resample
    particles = low_variance_resample(particles, weights)

    if step in snap_steps:
        snapshots_pf.append((f'Step {step + 1}', particles.copy(), true_state.copy()))

# Plot snapshots
n_snaps = len(snapshots_pf)
fig, axes = plt.subplots(1, n_snaps, figsize=(4.5 * n_snaps, 4.5))

for ax, (label, pts, ts) in zip(axes, snapshots_pf):
    ax.scatter(pts[:, 0], pts[:, 1], s=3, alpha=0.4, color='steelblue', zorder=3)
    ax.plot(ts[0], ts[1], 'r*', markersize=16, markeredgecolor='k', markeredgewidth=0.5, zorder=5)
    for lm in landmarks_demo:
        ax.plot(lm[0], lm[1], 's', color='orange', markersize=8, markeredgecolor='k', zorder=5)
    ax.set_xlim(-1, room_size + 1); ax.set_ylim(-1, room_size + 1)
    ax.set_aspect('equal')
    ax.set_title(label, fontsize=10)

fig.suptitle('Particle Filter: from uniform scatter to convergence\n'
             '(blue dots = particles, red star = true position, orange squares = landmarks)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

est = particles[:, :2].mean(axis=0)
print(f'True position:     ({true_state[0]:.2f}, {true_state[1]:.2f})')
print(f'Particle estimate: ({est[0]:.2f}, {est[1]:.2f})')
print(f'Error:             {np.linalg.norm(est - true_state[:2]):.3f} m')

## 18.1 Sampling Based Belief

Instead of representing the belief as a parametric distribution (like a Gaussian in Kalman filters) or a grid (like discrete Bayes filters), a particle filter represents the belief as a **set of weighted samples**:

$$\text{bel}(\mathbf{x}) \approx \{(\mathbf{x}^{[i]}, w^{[i]})\}_{i=1}^{N}$$

where $\mathbf{x}^{[i]}$ is the $i$th particle's state and $w^{[i]}$ is its weight, with $\sum_i w^{[i]} = 1$.

**Advantages** of this representation:
- Can represent **any** distribution shape (multimodal, skewed, heavy-tailed)
- No linearity or Gaussianity assumptions
- Trivially handles nonlinear dynamics: just propagate each particle through $g(\cdot)$

**Disadvantages:**
- Accuracy scales with the number of particles $N$
- Computational cost is $O(N)$ per step
- In high dimensions, you need exponentially many particles (the **curse of dimensionality**)

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_repr = [20, 100, 500, 2000]  # compare different counts
distribution_type = 'bimodal'             # 'gaussian', 'bimodal', 'ring'
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

def sample_distribution(n, dist_type):
    if dist_type == 'gaussian':
        return np.random.multivariate_normal([5, 5], [[1, 0.5], [0.5, 1]], n)
    elif dist_type == 'bimodal':
        n1 = n // 2
        n2 = n - n1
        s1 = np.random.multivariate_normal([3, 3], [[0.5, 0], [0, 0.5]], n1)
        s2 = np.random.multivariate_normal([8, 7], [[0.3, 0.2], [0.2, 0.8]], n2)
        return np.vstack([s1, s2])
    elif dist_type == 'ring':
        angles = np.random.uniform(0, 2*np.pi, n)
        radii = 3.0 + np.random.normal(0, 0.3, n)
        return np.column_stack([5 + radii * np.cos(angles), 5 + radii * np.sin(angles)])
    return np.random.uniform(0, 10, (n, 2))

fig, axes = plt.subplots(1, len(n_particles_repr), figsize=(4 * len(n_particles_repr), 4))

for ax, n_p in zip(axes, n_particles_repr):
    samples = sample_distribution(n_p, distribution_type)
    ax.scatter(samples[:, 0], samples[:, 1], s=max(1, 50 // (n_p // 20 + 1)),
               alpha=0.5, color='steelblue')
    ax.set_xlim(0, 11); ax.set_ylim(0, 11)
    ax.set_aspect('equal')
    ax.set_title(f'N = {n_p}', fontsize=11)

fig.suptitle(f'Particle Representation of a {distribution_type} distribution\n'
             'More particles = better approximation',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('With few particles, the shape is barely recognizable.')
print('With many particles, the distribution structure becomes clear.')

## 18.2 Importance Sampling

The core idea behind particle filtering is **importance sampling**. We cannot sample directly from the true posterior $p(\mathbf{x}_t \mid \mathbf{z}_{1:t})$, so instead we sample from a simpler **proposal distribution** $q(\mathbf{x}_t)$ and correct for the mismatch using **importance weights**:

$$w^{[i]} = \frac{p(\mathbf{z}_t \mid \mathbf{x}_t^{[i]})\, p(\mathbf{x}_t^{[i]} \mid \mathbf{x}_{t-1}^{[i]}, \mathbf{u}_t)}{q(\mathbf{x}_t^{[i]})}$$

In the simplest (and most common) form, the proposal distribution is the motion model itself: $q(\mathbf{x}_t^{[i]}) = p(\mathbf{x}_t^{[i]} \mid \mathbf{x}_{t-1}^{[i]}, \mathbf{u}_t)$. This cancels the denominator, leaving:

$$w^{[i]} = p(\mathbf{z}_t \mid \mathbf{x}_t^{[i]})$$

That is, the weight of each particle is simply the **likelihood** of the current measurement given that particle's state. Particles that explain the measurement well get high weight; particles that do not get low weight.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_is  = 500      # number of particles           (try 100, 500, 2000)
true_pos_is     = np.array([6.0, 7.0])  # true robot position
lm_is           = np.array([[2, 2], [10, 10], [2, 10], [10, 2]])  # landmarks
sensor_noise_is = 0.8      # range sensor noise std (m)    (try 0.2, 0.8, 2.0)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(7)

# Scatter particles uniformly
parts_is = np.random.uniform(0, 12, (n_particles_is, 2))

# True measurement
z_true_is = np.sqrt(np.sum((lm_is - true_pos_is)**2, axis=1))
z_meas_is = z_true_is + np.random.normal(0, sensor_noise_is, len(lm_is))

# Compute weights (likelihood)
weights_is = np.zeros(n_particles_is)
for i in range(n_particles_is):
    z_pred = np.sqrt(np.sum((lm_is - parts_is[i])**2, axis=1))
    diff = z_meas_is - z_pred
    weights_is[i] = np.exp(-0.5 * np.sum((diff / sensor_noise_is)**2))

weights_is += 1e-300
weights_is /= weights_is.sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before weighting (uniform)
axes[0].scatter(parts_is[:, 0], parts_is[:, 1], s=10, alpha=0.5, color='steelblue')
axes[0].plot(true_pos_is[0], true_pos_is[1], 'r*', markersize=16, markeredgecolor='k', zorder=5)
for lm in lm_is:
    axes[0].plot(lm[0], lm[1], 's', color='orange', markersize=10, markeredgecolor='k', zorder=5)
axes[0].set_xlim(-1, 13); axes[0].set_ylim(-1, 13); axes[0].set_aspect('equal')
axes[0].set_title('Before weighting: all particles equal', fontsize=11)

# After weighting (size proportional to weight)
sizes = weights_is / weights_is.max() * 100 + 1
axes[1].scatter(parts_is[:, 0], parts_is[:, 1], s=sizes, alpha=0.6, color='steelblue')
axes[1].plot(true_pos_is[0], true_pos_is[1], 'r*', markersize=16, markeredgecolor='k', zorder=5)
for lm in lm_is:
    axes[1].plot(lm[0], lm[1], 's', color='orange', markersize=10, markeredgecolor='k', zorder=5)
axes[1].set_xlim(-1, 13); axes[1].set_ylim(-1, 13); axes[1].set_aspect('equal')
axes[1].set_title('After weighting: size = importance weight', fontsize=11)

fig.suptitle('Importance Sampling: particles near true position get higher weights',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

weighted_mean = np.average(parts_is, weights=weights_is, axis=0)
print(f'Weighted mean estimate: ({weighted_mean[0]:.2f}, {weighted_mean[1]:.2f})')
print(f'True position:         ({true_pos_is[0]:.2f}, {true_pos_is[1]:.2f})')
print(f'Error:                 {np.linalg.norm(weighted_mean - true_pos_is):.3f} m')
print(f'Max weight:            {weights_is.max():.6f}')
print(f'Effective sample size: {1.0 / np.sum(weights_is**2):.1f} / {n_particles_is}')

## 18.3 Weight Update

The **weight update** step is the heart of the particle filter. For each particle $\mathbf{x}^{[i]}$, we compute the likelihood of the actual measurement $\mathbf{z}_t$ given that particle's state:

$$w^{[i]} \propto p(\mathbf{z}_t \mid \mathbf{x}_t^{[i]})$$

For range measurements with Gaussian noise $\sigma_r$:

$$p(z_k \mid \mathbf{x}) = \frac{1}{\sqrt{2\pi}\sigma_r} \exp\left(-\frac{(z_k - \hat{z}_k(\mathbf{x}))^2}{2\sigma_r^2}\right)$$

where $\hat{z}_k(\mathbf{x})$ is the predicted range from state $\mathbf{x}$ to landmark $k$.

For multiple independent landmarks, multiply the likelihoods (or sum the log-likelihoods):

$$w^{[i]} \propto \prod_k p(z_k \mid \mathbf{x}^{[i]})$$

After computing all weights, **normalize** them so they sum to 1.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_wu  = 1000     # number of particles        (try 200, 1000, 5000)
sensor_noise_wu = 0.5      # sensor noise std           (try 0.2, 0.5, 2.0)
n_landmarks_wu  = 3        # how many landmarks to use  (try 1, 2, 3, 4)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

all_lms = np.array([[2, 2], [2, 10], [10, 2], [10, 10]])
lms_wu = all_lms[:n_landmarks_wu]
true_pos_wu = np.array([6.0, 6.0])

# Particles scattered around
parts_wu = np.random.uniform(0, 12, (n_particles_wu, 2))

# Compute weights for 1 landmark, 2 landmarks, and all landmarks
z_true_wu = np.sqrt(np.sum((lms_wu - true_pos_wu)**2, axis=1))
z_meas_wu = z_true_wu + np.random.normal(0, sensor_noise_wu, n_landmarks_wu)

fig, axes = plt.subplots(1, n_landmarks_wu, figsize=(5 * n_landmarks_wu, 5))
if n_landmarks_wu == 1:
    axes = [axes]

cumulative_log_w = np.zeros(n_particles_wu)

for ax_idx, ax in enumerate(axes):
    k = ax_idx  # which landmark to add
    z_pred = np.sqrt(np.sum((lms_wu[k] - parts_wu)**2, axis=1))
    log_lik = -0.5 * ((z_meas_wu[k] - z_pred) / sensor_noise_wu)**2
    cumulative_log_w += log_lik

    w = np.exp(cumulative_log_w - cumulative_log_w.max())
    w /= w.sum()

    sizes = w / w.max() * 80 + 1
    ax.scatter(parts_wu[:, 0], parts_wu[:, 1], s=sizes, alpha=0.5, color='steelblue')
    ax.plot(true_pos_wu[0], true_pos_wu[1], 'r*', markersize=16, markeredgecolor='k', zorder=5)
    for j in range(ax_idx + 1):
        ax.plot(lms_wu[j, 0], lms_wu[j, 1], 's', color='orange', markersize=10,
                markeredgecolor='k', zorder=5)
    ax.set_xlim(-1, 13); ax.set_ylim(-1, 13); ax.set_aspect('equal')
    n_eff = 1.0 / np.sum(w**2)
    ax.set_title(f'After {ax_idx + 1} landmark(s), Neff = {n_eff:.0f}', fontsize=10)

fig.suptitle('Weight Update: each landmark sharpens the weight distribution',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('With 1 landmark: weights form a ring (range is ambiguous in direction).')
print('With 2+ landmarks: the intersection localizes the robot.')
print('More landmarks = tighter localization.')

## 18.4 Resampling

After a few weight update steps, most particles will have negligible weight while a few dominate. This means we are wasting computation on particles that contribute nothing to the estimate. **Resampling** fixes this by duplicating high-weight particles and discarding low-weight ones.

The **low-variance resampling** algorithm (also called **systematic resampling**) is the standard choice. It draws $N$ new particles from the old set, with probability proportional to weight:

1. Draw a single random number $r \sim \text{Uniform}(0, 1/N)$
2. Walk through the cumulative weight distribution with evenly spaced pointers at $r, r + 1/N, r + 2/N, \dots$
3. For each pointer, select the particle whose cumulative weight interval contains that pointer

This is $O(N)$ and produces less variance than multinomial resampling.

**Particle impoverishment:** after resampling, many particles are duplicates. Over time, this reduces diversity. If the process noise is small, particles may collapse to a single point and lose track of the true state.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_rs  = 30       # small number to see individual particles (try 15, 30, 50)
weights_skew    = 5.0      # how skewed the weights are (try 1.0, 3.0, 5.0, 10.0)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# Create particles on a line
x_pos = np.linspace(1, 10, n_particles_rs)
y_pos = np.ones(n_particles_rs) * 5
particles_rs = np.column_stack([x_pos, y_pos])

# Create skewed weights (favoring higher x values)
raw_w = np.exp(weights_skew * (x_pos - x_pos.mean()) / x_pos.std())
weights_rs = raw_w / raw_w.sum()

# Low-variance resampling
def low_variance_resample_viz(parts, w):
    N = len(w)
    r = np.random.uniform(0, 1.0 / N)
    cumw = np.cumsum(w)
    pointers = r + np.arange(N) / N
    indices = np.searchsorted(cumw, pointers)
    return parts[indices].copy(), indices, pointers, cumw

resampled, idx, ptrs, cw = low_variance_resample_viz(particles_rs, weights_rs)

fig, axes = plt.subplots(3, 1, figsize=(12, 8))

# Before resampling with weights
ax = axes[0]
sizes_b = weights_rs / weights_rs.max() * 300 + 10
ax.scatter(particles_rs[:, 0], particles_rs[:, 1], s=sizes_b, color='steelblue',
           edgecolor='white', zorder=3)
for i in range(n_particles_rs):
    ax.annotate(f'{weights_rs[i]:.3f}', (particles_rs[i, 0], particles_rs[i, 1] + 0.3),
                ha='center', fontsize=6, color='steelblue')
ax.set_xlim(0, 11); ax.set_ylim(3.5, 6.5)
ax.set_title('Before resampling: size = weight', fontsize=11, loc='left')
ax.set_yticks([])

# Cumulative weight and pointers
ax = axes[1]
ax.step(x_pos, cw, 'steelblue', lw=2, where='post', label='Cumulative weight')
for p in ptrs:
    ax.axhline(p, color='tomato', lw=0.5, alpha=0.5)
ax.scatter(x_pos[idx], ptrs, color='tomato', s=30, zorder=5, label='Selected')
ax.set_xlim(0, 11)
ax.set_title('Low-variance resampling: evenly spaced pointers on cumulative weight',
             fontsize=10, loc='left')
ax.set_ylabel('Cumulative weight')
ax.legend(fontsize=8)

# After resampling
ax = axes[2]
# Count duplicates
unique, counts = np.unique(idx, return_counts=True)
sizes_a = np.zeros(n_particles_rs)
for u, c in zip(unique, counts):
    sizes_a[u] = c

# Jitter resampled particles vertically for visibility
jittered = resampled.copy()
for i in range(len(resampled)):
    jittered[i, 1] += np.random.normal(0, 0.15)

ax.scatter(jittered[:, 0], jittered[:, 1], s=50, color='forestgreen',
           edgecolor='white', alpha=0.7, zorder=3)
ax.set_xlim(0, 11); ax.set_ylim(3.5, 6.5)
ax.set_title('After resampling: particles duplicated/removed (all weights now equal)',
             fontsize=10, loc='left')
ax.set_yticks([])
ax.set_xlabel('x position')

plt.tight_layout()
plt.show()

n_unique = len(np.unique(idx))
print(f'Unique particles after resampling: {n_unique} / {n_particles_rs}')
print(f'Particle impoverishment: {n_particles_rs - n_unique} particles were duplicates.')

## 18.5 Degeneracy and Effective Sample Size

**Weight degeneracy** is the fundamental problem of particle filters. After several update steps without resampling, the weights become extremely skewed: one or two particles have almost all the weight while the rest are effectively dead.

The **effective sample size** (ESS) quantifies this:

$$N_{\text{eff}} = \frac{1}{\sum_{i=1}^{N} (w^{[i]})^2}$$

- If all weights are equal: $N_{\text{eff}} = N$ (perfect)
- If one particle has all weight: $N_{\text{eff}} = 1$ (degenerate)

**Common strategy:** resample when $N_{\text{eff}} < N/2$. This balances between resampling too often (losing diversity) and too rarely (wasting computation on dead particles).

**Particle depletion** occurs when the filter runs out of particles in the right region. This can happen when:
- The true state moves to a region with no particles
- The sensor is very precise and no particles happen to be close enough
- The state space is large and the number of particles is too small

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_deg = 500       # number of particles          (try 100, 500, 2000)
n_updates_deg   = 15        # number of weight updates     (try 5, 10, 15, 25)
sensor_noise_deg = 0.5      # sensor noise                 (try 0.2, 0.5, 1.5)
resample_threshold = 0.5    # resample when Neff < threshold * N  (try 0.3, 0.5, 0.8)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

true_pos_deg = np.array([6.0, 6.0])
lm_deg = np.array([[2, 2], [10, 10], [2, 10]])

# Run with and without resampling
results = {}
for do_resample, label in [(False, 'No resampling'), (True, 'With resampling')]:
    np.random.seed(42)
    parts = np.random.uniform(0, 12, (n_particles_deg, 2))
    w = np.ones(n_particles_deg) / n_particles_deg
    neff_history = [n_particles_deg]
    max_w_history = [1.0 / n_particles_deg]

    for step in range(n_updates_deg):
        # Slight motion of true position
        true_pos_step = true_pos_deg + np.array([0.1 * step, 0.05 * step])

        # Simulated measurement
        z_t = np.sqrt(np.sum((lm_deg - true_pos_step)**2, axis=1))
        z_noisy = z_t + np.random.normal(0, sensor_noise_deg, len(lm_deg))

        # Weight update
        for i in range(n_particles_deg):
            z_pred = np.sqrt(np.sum((lm_deg - parts[i])**2, axis=1))
            diff = z_noisy - z_pred
            log_lik = -0.5 * np.sum((diff / sensor_noise_deg)**2)
            w[i] *= np.exp(log_lik)

        w += 1e-300
        w /= w.sum()

        neff = 1.0 / np.sum(w**2)
        neff_history.append(neff)
        max_w_history.append(w.max())

        if do_resample and neff < resample_threshold * n_particles_deg:
            parts = low_variance_resample(parts, w)
            w = np.ones(n_particles_deg) / n_particles_deg

    results[label] = (neff_history, max_w_history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for label, (neff_h, _) in results.items():
    color = 'tomato' if 'No' in label else 'steelblue'
    ax.plot(range(len(neff_h)), neff_h, color=color, lw=2, marker='o', markersize=4, label=label)
ax.axhline(resample_threshold * n_particles_deg, color='orange', ls='--', lw=1.5,
           label=f'Resample threshold ({resample_threshold * n_particles_deg:.0f})')
ax.set_xlabel('Update step'); ax.set_ylabel('$N_{eff}$')
ax.set_title('Effective Sample Size Over Time'); ax.legend(fontsize=9)

ax = axes[1]
for label, (_, maxw_h) in results.items():
    color = 'tomato' if 'No' in label else 'steelblue'
    ax.plot(range(len(maxw_h)), maxw_h, color=color, lw=2, marker='o', markersize=4, label=label)
ax.set_xlabel('Update step'); ax.set_ylabel('Max weight')
ax.set_title('Maximum Particle Weight (closer to 1 = more degenerate)'); ax.legend(fontsize=9)

fig.suptitle('Degeneracy: without resampling, one particle takes all the weight',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

for label, (neff_h, _) in results.items():
    print(f'{label}: final Neff = {neff_h[-1]:.1f} / {n_particles_deg}')

## Capstone: Monte Carlo Localization with Kidnapping Recovery

This is the full Monte Carlo Localization (MCL) algorithm. A robot moves through a 2D room with walls and range sensors. We track it with 500 particles. After the filter converges, we **kidnap** the robot (teleport it to a new location) and watch the filter fail. Then we add **random particle injection** (replacing a small fraction of particles with uniformly random samples at each step) and show that this enables recovery.

This is the standard approach used in production robot localization systems.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_particles_mcl   = 500      # number of particles            (try 200, 500, 1000)
room_w, room_h    = 20, 15   # room dimensions
lm_mcl = np.array([[2, 2], [2, 13], [18, 2], [18, 13], [10, 7]])  # 5 landmarks
sensor_noise_mcl  = 1.0      # range noise std (m)            (try 0.3, 1.0, 2.0)
motion_noise_mcl  = 0.4      # motion noise std (m)           (try 0.1, 0.4, 1.0)
heading_noise_mcl = 0.1      # heading noise std (rad)        (try 0.02, 0.1, 0.3)
kidnap_step       = 30       # when to kidnap                 (try 20, 30, 40)
kidnap_to         = np.array([16.0, 12.0])  # where to teleport
n_total_mcl       = 80       # total steps                    (try 50, 80, 120)
random_inject_frac = 0.0     # fraction of random particles   (try 0.0, 0.02, 0.05)
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(2024)

def mcl_run(inject_frac, label):
    """Run full MCL with optional random particle injection."""
    # Initialize
    parts = np.zeros((n_particles_mcl, 3))  # [x, y, theta]
    parts[:, 0] = np.random.uniform(0, room_w, n_particles_mcl)
    parts[:, 1] = np.random.uniform(0, room_h, n_particles_mcl)
    parts[:, 2] = np.random.uniform(-np.pi, np.pi, n_particles_mcl)

    true_state = np.array([4.0, 4.0, 0.3])
    v_mcl = 0.8

    errors = []
    snapshots = []
    snap_at = [0, 10, kidnap_step - 1, kidnap_step + 1, kidnap_step + 5,
               kidnap_step + 15, n_total_mcl - 1]
    snap_at = sorted(set(s for s in snap_at if s < n_total_mcl))

    for step in range(n_total_mcl):
        # Kidnap
        if step == kidnap_step:
            true_state[:2] = kidnap_to.copy()
            true_state[2] = -1.0

        # Move true robot
        true_state[0] += v_mcl * np.cos(true_state[2])
        true_state[1] += v_mcl * np.sin(true_state[2])
        true_state[2] += 0.08
        # Keep in room
        true_state[0] = np.clip(true_state[0], 0.5, room_w - 0.5)
        true_state[1] = np.clip(true_state[1], 0.5, room_h - 0.5)

        # Move particles
        parts[:, 2] += 0.08 + np.random.normal(0, heading_noise_mcl, n_particles_mcl)
        parts[:, 0] += v_mcl * np.cos(parts[:, 2]) + np.random.normal(0, motion_noise_mcl, n_particles_mcl)
        parts[:, 1] += v_mcl * np.sin(parts[:, 2]) + np.random.normal(0, motion_noise_mcl, n_particles_mcl)

        # Random injection
        if inject_frac > 0:
            n_inject = max(1, int(inject_frac * n_particles_mcl))
            idx_inject = np.random.choice(n_particles_mcl, n_inject, replace=False)
            parts[idx_inject, 0] = np.random.uniform(0, room_w, n_inject)
            parts[idx_inject, 1] = np.random.uniform(0, room_h, n_inject)
            parts[idx_inject, 2] = np.random.uniform(-np.pi, np.pi, n_inject)

        # Sense: range to landmarks
        z_true = np.sqrt(np.sum((lm_mcl - true_state[:2])**2, axis=1))
        z_meas = z_true + np.random.normal(0, sensor_noise_mcl, len(lm_mcl))

        # Weight
        log_weights = np.zeros(n_particles_mcl)
        for i in range(n_particles_mcl):
            z_pred = np.sqrt(np.sum((lm_mcl - parts[i, :2])**2, axis=1))
            diff = z_meas - z_pred
            log_weights[i] = -0.5 * np.sum((diff / sensor_noise_mcl)**2)

        # Normalize weights
        log_weights -= log_weights.max()
        weights = np.exp(log_weights)
        weights += 1e-300
        weights /= weights.sum()

        # Estimate
        est = np.average(parts[:, :2], weights=weights, axis=0)
        err = np.linalg.norm(est - true_state[:2])
        errors.append(err)

        # Resample
        neff = 1.0 / np.sum(weights**2)
        if neff < n_particles_mcl * 0.5:
            parts = low_variance_resample(parts, weights)

        if step in snap_at:
            snapshots.append((step, parts.copy(), true_state.copy(), err))

    return errors, snapshots

# Run without injection
errors_no, snaps_no = mcl_run(0.0, 'No injection')
# Run with injection
errors_yes, snaps_yes = mcl_run(0.05, 'With injection (5%)')

# Plot snapshots for both cases
n_snaps_show = min(6, len(snaps_no))
fig, axes = plt.subplots(2, n_snaps_show, figsize=(4 * n_snaps_show, 8))

for row, (snaps, row_label) in enumerate([(snaps_no, 'Without injection'), (snaps_yes, 'With 5% injection')]):
    for col in range(n_snaps_show):
        ax = axes[row, col]
        if col < len(snaps):
            step, pts, ts, err = snaps[col]
            ax.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.4, color='steelblue')
            ax.plot(ts[0], ts[1], 'r*', markersize=14, markeredgecolor='k', zorder=5)
            for lm in lm_mcl:
                ax.plot(lm[0], lm[1], 's', color='orange', markersize=6, markeredgecolor='k', zorder=5)
            ax.set_xlim(-1, room_w + 1); ax.set_ylim(-1, room_h + 1)
            ax.set_aspect('equal')
            kidnap_label = ' [KIDNAPPED]' if step == kidnap_step + 1 else ''
            ax.set_title(f'Step {step}{kidnap_label}\nerr={err:.2f}m', fontsize=9)
        else:
            ax.set_visible(False)
        if col == 0:
            ax.set_ylabel(row_label, fontsize=10, fontweight='bold')

fig.suptitle('MCL: Kidnapping Recovery Comparison\n'
             f'(kidnap at step {kidnap_step})',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
inject_fracs_compare = [0.0, 0.02, 0.05, 0.10]  # injection fractions to compare
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 5))
colors_cmp = ['tomato', 'orange', 'steelblue', 'forestgreen']

for frac, color in zip(inject_fracs_compare, colors_cmp):
    np.random.seed(2024)
    errs, _ = mcl_run(frac, f'{frac*100:.0f}%')
    lbl = f'{frac*100:.0f}% injection' if frac > 0 else 'No injection'
    ax.plot(range(len(errs)), errs, color=color, lw=1.5, alpha=0.8, label=lbl)

ax.axvline(kidnap_step, color='k', ls='--', lw=1.5, alpha=0.5, label=f'Kidnap at step {kidnap_step}')
ax.set_xlabel('Step')
ax.set_ylabel('Position error (m)')
ax.set_title('MCL Tracking Error: Effect of Random Particle Injection on Kidnapping Recovery')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'Without injection: the filter stays lost after kidnapping.')
print(f'With injection: random particles eventually land near the true position and take over.')
print(f'More injection = faster recovery but slightly noisier tracking before kidnap.')

## Exercises

**Exercise 18.1: Implement low-variance resampling from scratch.**

Write the low-variance (systematic) resampling algorithm without using `np.searchsorted`. Your function should:
1. Take an array of weights (summing to 1) and return an array of selected indices
2. Draw a single random number $r \in [0, 1/N)$
3. Walk through the cumulative weight distribution with a running index

Verify your implementation matches `np.searchsorted` results on random weight vectors. Test with $N = 10$ and $N = 1000$.

In [ ]:
# Your code here


**Exercise 18.2: Particle count vs. accuracy.**

Run the opening demo (2D room with 4 landmarks) with particle counts $N = 10, 50, 100, 500, 2000, 5000$. For each, run 20 trials with different random seeds and record the final position error. Plot the mean and standard deviation of the error versus $N$ on a log-log plot. What is the approximate relationship between $N$ and error?

In [ ]:
# Your code here


**Exercise 18.3: Multinomial vs. systematic resampling.**

Implement **multinomial resampling** (draw $N$ independent samples with replacement from the weight distribution). Compare it with low-variance resampling on the same problem:
1. Run 50 trials of each on the opening demo
2. Record the position error over time for both
3. Compare the variance of the estimate

Why does low-variance resampling generally produce lower variance?

In [ ]:
# Your code here


**Exercise 18.4: Adaptive resampling.**

Instead of resampling every step, implement adaptive resampling: only resample when $N_{\text{eff}} < \alpha N$ for threshold $\alpha$. Test with $\alpha = 0.3, 0.5, 0.8, 1.0$ (where 1.0 means "always resample"). Plot:
1. How often resampling occurs for each $\alpha$
2. The tracking error over time
3. The number of unique particles over time

What is the best $\alpha$ for this problem?

In [ ]:
# Your code here


**Exercise 18.5 (Challenge): Particle filter vs. EKF comparison.**

Implement both a particle filter (500 particles) and an EKF for the same range-bearing landmark tracking problem used in Chapter 17. Use the same motion model, landmarks, and noise parameters. Run both for 200 steps and compare:
1. RMSE over time
2. Consistency (are the true states within the estimated uncertainty?)
3. What happens when the initial position estimate is very wrong (e.g., 10 meters off)?

The particle filter should handle the wrong initialization better because it can maintain a multimodal belief.

In [ ]:
# Your code here
